In [ ]:
pip install selenium beautifulsoup4 pandas tqdm
pip install webdriver_manager

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import time

# ✅ 셀레니움 옵션 설정
options = Options()
options.add_argument('--headless=new')  # 필요시 주석 해제
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--no-sandbox')
options.add_argument('--disable-gpu')
options.add_argument('--blink-settings=imagesEnabled=false')

# ✅ 법정동 리스트 불러오기
df_dong = pd.read_csv("법정동_리스트 - 복사본.csv")
dong_list = df_dong[["법정동코드", "법정동"]].drop_duplicates().values.tolist()

all_results = []
failed_dongs = []

# ✅ 단지 상세정보 병렬 처리 함수
def crawl_apt_detail(apt, dong_name):
    result = {
        "법정동": dong_name,
        "단지명": apt["name"],
        "카테고리": apt["category"],
        "주소": None,
        "요약글": "요약글 없음",
        "해시태그": ""
    }
    try:
        driver = webdriver.Chrome(options=options)
        wait = WebDriverWait(driver, 10)
        driver.get(apt["url"])
        time.sleep(2)

        # 주소
        try:
            addr = wait.until(EC.presence_of_element_located(
                (By.CSS_SELECTOR, "div.text-sm.font-semibold.text-foreground")
            )).text.strip()
            result["주소"] = addr
        except:
            pass

        # 요약 및 해시태그
        try:
            top_button = wait.until(
                EC.element_to_be_clickable((By.CSS_SELECTOR,
                    "#page-header > div > div.absolute.bottom-0.left-0.right-0.h-12.bg-primary.px-5.z-layer.text-background > div > div > div > div.flex.flex-1 > button"))
            )
            driver.execute_script("arguments[0].click();", top_button)
            time.sleep(1.5)

            summaries = driver.find_elements(By.CSS_SELECTOR, "p.px-5.text-base.text-foreground")
            if summaries:
                result["요약글"] = summaries[0].text.strip()

            tag_elements = driver.find_elements(By.CSS_SELECTOR,
                "#reviewPage-scroll > div > section:nth-child(3) > div.mt-4.flex.flex-wrap.gap-2.px-5 > a")
            tags = [tag.text.strip() for tag in tag_elements]
            result["해시태그"] = ", ".join(tags)
        except:
            pass

        driver.quit()
    except:
        pass
    return result

# ✅ 동별 반복
for code, dong_name in dong_list:
    try:
        # 단지 리스트 크롤링
        driver = webdriver.Chrome(options=options)
        wait = WebDriverWait(driver, 10)
        url = f"https://hogangnono.com/region/{code}/0/apt-list"
        driver.get(url)
        time.sleep(3)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        section = soup.select_one("#local3-aptlist-scroll")
        if not section:
            print(f"[❌ 없음] {dong_name} (코드: {code}) - 리스트 영역 없음")
            failed_dongs.append((code, dong_name))
            driver.quit()
            continue

        category_blocks = section.select("div.css-jsrvbw.ekplin50")
        apt_list = []

        for block in category_blocks:
            category = block.select_one("h3.type").text.strip()
            links = block.select("ul.region-list li a")
            for link in links:
                name_tag = link.select_one("h5")
                href = link.get("href")
                if name_tag and href:
                    apt_list.append({
                        "name": name_tag.text.strip(),
                        "url": f"https://hogangnono.com{href}",
                        "category": category
                    })

        driver.quit()

        # 단지 상세정보 멀티스레드 병렬 크롤링
        print(f"▶ 단지 수집 완료: {dong_name} ({len(apt_list)}개), 상세정보 수집 중...")
        with ThreadPoolExecutor(max_workers=5) as executor:
            futures = [executor.submit(crawl_apt_detail, apt, dong_name) for apt in apt_list]
            for future in as_completed(futures):
                result = future.result()
                all_results.append(result)

        print(f"✅ 완료: {dong_name} ({code})")

    except Exception as e:
        print(f"[❌ 실패] {dong_name} (코드: {code}) - {e}")
        failed_dongs.append((code, dong_name))
        continue

# ✅ 저장
df_all = pd.DataFrame(all_results)
df_all.to_csv("서울시_호갱노노.csv", index=False, encoding="utf-8-sig")

# ✅ 실패 목록 출력
if failed_dongs:
    print("\n❌ 크롤링 실패한 동 목록:")
    for code, name in failed_dongs:
        print(f"- {name} ({code})")
else:
    print("\n✅ 모든 법정동 크롤링 성공!")

▶ 단지 수집 완료: 개포동 (48개), 상세정보 수집 중...
✅ 완료: 개포동 (11680103)
▶ 단지 수집 완료: 일원동 (18개), 상세정보 수집 중...
✅ 완료: 일원동 (11680114)

✅ 모든 법정동 크롤링 성공!
